# Notebook 3 — Vector DB, chunking e reranking

Objetivos:

- carregar um pequeno corpus de políticas internas
- comparar chunking ingênuo vs chunking melhor
- indexar localmente em Chroma
- recuperar candidatos com busca vetorial
- reranquear os resultados
- mostrar como a mesma lógica troca de backend


## Stack do demo

- embeddings: local por default, OpenAI opcional
- vector store executado ao vivo: `Chroma`
- reranker local: `sentence-transformers` com cross-encoder

O notebook também inclui snippets de uso para `pgvector`, `Qdrant`, `Pinecone`, `Weaviate` e `Milvus`.


In [16]:
import os
import shutil
import warnings
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from IPython.display import Markdown, display
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import CrossEncoder, SentenceTransformer

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 160)


In [17]:
cwd = Path.cwd()
candidates = [
    cwd,
    cwd.parent,
    cwd / "aulao-3-embeddings-e-vector-db",
    cwd.parent / "aulao-3-embeddings-e-vector-db",
]
if cwd.name == "notebooks":
    candidates.insert(0, cwd.parent)
PROJECT_ROOT = next(
    (
        path
        for path in candidates
        if (path / ".env").exists() or (path / "data" / "rag_docs").exists()
    ),
    cwd.parent if cwd.name == "notebooks" else cwd,
)
load_dotenv(PROJECT_ROOT / ".env", override=True)

DATA_DIR = PROJECT_ROOT / "data" / "rag_docs"
doc_paths = sorted(DATA_DIR.glob("*.md"))

raw_docs = [
    {
        "source": path.stem,
        "text": path.read_text(encoding="utf-8"),
    }
    for path in doc_paths
]

pd.DataFrame(
    {
        "source": [doc["source"] for doc in raw_docs],
        "chars": [len(doc["text"]) for doc in raw_docs],
        "preview": [doc["text"][:110].replace("\n", " ") + "..." for doc in raw_docs],
    }
)


,source,chars,preview
0,01_politica_reembolso,1528,# Politica de Reembolso e Despesas ## Objetivo Esta politica define quais gastos corporativos podem ser reem...
1,02_politica_viagens,1057,# Politica de Viagens Corporativas ## Planejamento Solicitacoes de viagem devem ser registradas com pelo men...
2,03_politica_home_office,776,"# Politica de Home Office ## Elegibilidade Times de produto, engenharia, dados e operacoes podem operar em m..."
3,04_manual_compras,847,"# Manual de Compras e Contratacao ## Objetivo Este manual define como comprar software, hardware e servicos ..."
4,05_incidentes_seguranca,821,"# Runbook de Incidentes de Seguranca ## Severidades - Sev 1: vazamento em andamento, indisponibilidade criti..."


In [18]:
display(Markdown(raw_docs[0]["text"][:1000]))


# Politica de Reembolso e Despesas

## Objetivo

Esta politica define quais gastos corporativos podem ser reembolsados, quais comprovantes sao obrigatorios e qual e o fluxo de aprovacao.

## Despesas elegiveis

- Taxi ou carro por aplicativo entre aeroporto, hotel e cliente.
- Almoco e jantar durante viagem aprovada.
- Pedagio e estacionamento vinculados a visita comercial.
- Compra emergencial de cabos, adaptadores e pequenos insumos de reuniao ate R$ 150.

## Limites padrao

- Almoco: ate R$ 65 por pessoa.
- Jantar: ate R$ 120 por pessoa.
- Transporte aeroporto-hotel: valor integral, desde que exista comprovante.
- Transporte urbano dentro da cidade do cliente: ate R$ 80 por trecho sem aprovacao adicional.

## Comprovantes e prazo

Todos os comprovantes devem ser enviados pelo portal financeiro em ate 5 dias uteis apos o retorno da viagem. Isso inclui recibos de taxi do aeroporto, notas de refeicao, estacionamento e pedagio.

Quando o colaborador perder o comprovante, a despesa pode 

## Chunking

Vamos comparar duas estratégias:

- `fixed-size`: corta o texto a cada N caracteres, ignorando estrutura
- `recursive`: tenta respeitar títulos, listas e quebras naturais

Em produção normalmente você começa com recursive e mede recall antes de sofisticar.


In [19]:
def base_documents():
    return [Document(page_content=doc["text"], metadata={"source": doc["source"]}) for doc in raw_docs]


def fixed_size_chunks(documents, chunk_size=160, overlap=20):
    chunks = []
    for doc in documents:
        text = doc.page_content
        start = 0
        chunk_id = 0
        while start < len(text):
            end = start + chunk_size
            chunk_text = text[start:end]
            chunks.append(
                Document(
                    page_content=chunk_text,
                    metadata={
                        **doc.metadata,
                        "chunking": "fixed",
                        "chunk_id": chunk_id,
                    },
                )
            )
            if end >= len(text):
                break
            start = end - overlap
            chunk_id += 1
    return chunks


docs = base_documents()
fixed_chunks = fixed_size_chunks(docs, chunk_size=160, overlap=20)

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=320,
    chunk_overlap=50,
    separators=["\n## ", "\n### ", "\n- ", "\n", ". ", " "],
)
recursive_chunks = recursive_splitter.split_documents(docs)
for idx, chunk in enumerate(recursive_chunks):
    chunk.metadata["chunking"] = "recursive"
    chunk.metadata["chunk_id"] = idx

chunk_stats = pd.DataFrame(
    [
        {
            "estrategia": "fixed",
            "chunks": len(fixed_chunks),
            "tamanho_medio_chars": round(sum(len(c.page_content) for c in fixed_chunks) / len(fixed_chunks), 1),
        },
        {
            "estrategia": "recursive",
            "chunks": len(recursive_chunks),
            "tamanho_medio_chars": round(
                sum(len(c.page_content) for c in recursive_chunks) / len(recursive_chunks), 1
            ),
        },
    ]
)

chunk_stats


,estrategia,chunks,tamanho_medio_chars
0,fixed,37,153.2
1,recursive,23,216.9


In [20]:
source_name = "01_politica_reembolso"

preview_rows = []
for strategy_name, chunks in [("fixed", fixed_chunks), ("recursive", recursive_chunks)]:
    selected = [chunk for chunk in chunks if chunk.metadata["source"] == source_name][:3]
    for chunk in selected:
        preview_rows.append(
            {
                "estrategia": strategy_name,
                "chunk_id": chunk.metadata["chunk_id"],
                "texto": chunk.page_content.replace("\n", " "),
            }
        )

pd.DataFrame(preview_rows)


,estrategia,chunk_id,texto
0,fixed,0,"# Politica de Reembolso e Despesas ## Objetivo Esta politica define quais gastos corporativos podem ser reembolsados, quais comprovantes sao obrigatorios ..."
1,fixed,1,"ao obrigatorios e qual e o fluxo de aprovacao. ## Despesas elegiveis - Taxi ou carro por aplicativo entre aeroporto, hotel e cliente. - Almoco e jantar du..."
2,fixed,2,"moco e jantar durante viagem aprovada. - Pedagio e estacionamento vinculados a visita comercial. - Compra emergencial de cabos, adaptadores e pequenos insum..."
3,recursive,0,"# Politica de Reembolso e Despesas ## Objetivo Esta politica define quais gastos corporativos podem ser reembolsados, quais comprovantes sao obrigatorios ..."
4,recursive,1,"## Despesas elegiveis - Taxi ou carro por aplicativo entre aeroporto, hotel e cliente. - Almoco e jantar durante viagem aprovada. - Pedagio e estacionament..."
5,recursive,2,"## Limites padrao - Almoco: ate R$ 65 por pessoa. - Jantar: ate R$ 120 por pessoa. - Transporte aeroporto-hotel: valor integral, desde que exista comprovan..."


## Embeddings

O mesmo notebook roda com:

- provider local via `SentenceTransformer`
- provider OpenAI via `OpenAIEmbeddings`

O backend vetorial não muda a lógica de ingestão: `chunk -> embed -> upsert`.


In [21]:
EMBEDDING_PROVIDER = "openai" if os.getenv("OPENAI_API_KEY") else "local"
LOCAL_MODEL = os.getenv(
    "LOCAL_EMBEDDING_MODEL",
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
)


class LocalSentenceTransformerEmbeddings:
    def __init__(self, model_name):
        self.model = SentenceTransformer(model_name)

    def embed_documents(self, texts):
        return self.model.encode(texts, normalize_embeddings=True).tolist()

    def embed_query(self, text):
        return self.model.encode([text], normalize_embeddings=True)[0].tolist()


if EMBEDDING_PROVIDER == "openai" and os.getenv("OPENAI_API_KEY"):
    from langchain_openai import OpenAIEmbeddings

    embeddings = OpenAIEmbeddings(
        model=os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")
    )
    provider_name = "openai"
else:
    embeddings = LocalSentenceTransformerEmbeddings(LOCAL_MODEL)
    provider_name = "local"

provider_name


'openai'

In [22]:
persist_root = (PROJECT_ROOT / ".chroma_lab").resolve()
persist_root.mkdir(parents=True, exist_ok=True)

# Usa diretórios novos a cada execução para evitar lock do SQLite ao rerodar a célula.
run_id = pd.Timestamp.now(tz="UTC").strftime("%Y%m%d_%H%M%S_%f")
fixed_dir = persist_root / f"fixed_{run_id}"
recursive_dir = persist_root / f"recursive_{run_id}"
fixed_dir.mkdir(parents=True, exist_ok=True)
recursive_dir.mkdir(parents=True, exist_ok=True)

fixed_store = Chroma(
    collection_name="policy_fixed",
    embedding_function=embeddings,
    persist_directory=str(fixed_dir),
    collection_metadata={"hnsw:space": "cosine"},
)
recursive_store = Chroma(
    collection_name="policy_recursive",
    embedding_function=embeddings,
    persist_directory=str(recursive_dir),
    collection_metadata={"hnsw:space": "cosine"},
)

fixed_store.add_documents(fixed_chunks)
recursive_store.add_documents(recursive_chunks)

{
    "fixed_chunks": len(fixed_chunks),
    "recursive_chunks": len(recursive_chunks),
    "persist_root": str(persist_root),
    "fixed_dir": str(fixed_dir),
    "recursive_dir": str(recursive_dir),
}


{'fixed_chunks': 37,
 'recursive_chunks': 23,
 'persist_root': '/Users/luisfelipegomesmolina/Documents/Github/bootcamp-ockm/aulao-3-embeddings-e-vector-db/.chroma_lab',
 'fixed_dir': '/Users/luisfelipegomesmolina/Documents/Github/bootcamp-ockm/aulao-3-embeddings-e-vector-db/.chroma_lab/fixed_20260415_233054_906796',
 'recursive_dir': '/Users/luisfelipegomesmolina/Documents/Github/bootcamp-ockm/aulao-3-embeddings-e-vector-db/.chroma_lab/recursive_20260415_233054_906796'}

In [8]:
sample_queries = [
    "Em quantos dias preciso enviar os comprovantes de taxi do aeroporto depois da viagem?",
    "Quem aprova compras de software acima de R$ 5.000?",
    "Com quanta antecedência devo pedir home office fora do acordo recorrente?",
]

sample_queries


['Em quantos dias preciso enviar os comprovantes de taxi do aeroporto depois da viagem?',
 'Quem aprova compras de software acima de R$ 5.000?',
 'Com quanta antecedência devo pedir home office fora do acordo recorrente?']

In [23]:
query = sample_queries[1]


def search(store, strategy_name, query_text, k=4):
    rows = []
    results = store.similarity_search_with_score(query_text, k=k)
    for rank, (doc, score) in enumerate(results, start=1):
        rows.append(
            {
                "estrategia": strategy_name,
                "rank": rank,
                "source": doc.metadata["source"],
                "chunk_id": doc.metadata["chunk_id"],
                "distance_lower_is_better": round(float(score), 4),
                "trecho": doc.page_content.replace("\n", " ")[:220],
            }
        )
    return pd.DataFrame(rows), results


fixed_results_df, fixed_results = search(fixed_store, "fixed", query)
recursive_results_df, recursive_results = search(recursive_store, "recursive", query)

pd.concat([fixed_results_df, recursive_results_df], ignore_index=True)


,estrategia,rank,source,chunk_id,distance_lower_is_better,trecho
0,fixed,1,04_manual_compras,2,0.3245,"00: diretor da area + compras. - Acima de R$ 20.000: comite de investimentos. ## Software Licencas de software SaaS devem ter owner interno, centro de cus..."
1,fixed,2,04_manual_compras,0,0.4005,"# Manual de Compras e Contratacao ## Objetivo Este manual define como comprar software, hardware e servicos com controle financeiro e compliance. ## Faix..."
2,fixed,3,04_manual_compras,4,0.4531,"fornecedores precisam enviar dados bancarios, contrato social e aceite do codigo de conduta. Compras sem cadastro previo nao devem ser executadas. ## Exce..."
3,fixed,4,01_politica_reembolso,7,0.4679,"nte, a despesa pode ser recusada. Excecoes so sao aceitas com justificativa escrita do gestor imediato e aprovacao do financeiro. ## Fluxo de aprovacao - ..."
4,recursive,1,04_manual_compras,16,0.4019,"# Manual de Compras e Contratacao ## Objetivo Este manual define como comprar software, hardware e servicos com controle financeiro e compliance."
5,recursive,2,04_manual_compras,19,0.4532,"## Fornecedores Novos fornecedores precisam enviar dados bancarios, contrato social e aceite do codigo de conduta. Compras sem cadastro previo nao devem se..."
6,recursive,3,04_manual_compras,18,0.4746,"## Software Licencas de software SaaS devem ter owner interno, centro de custo e prazo contratual definidos. Renovacoes automaticas precisam de revalidacao..."
7,recursive,4,04_manual_compras,17,0.5038,## Faixas de aprovacao - Ate R$ 1.000: lider de time. - De R$ 1.001 a R$ 5.000: gerente da area. - Acima de R$ 5.000 e ate R$ 20.000: diretor da area + com...


## Reranking

Estratégia padrão:

1. over-retrieve com vetor (`top_k` maior)
2. rerank com um modelo mais caro e mais preciso
3. mandar poucos chunks melhores para o LLM


In [24]:
candidate_docs = recursive_store.similarity_search(query, k=6)
candidate_df = pd.DataFrame(
    [
        {
            "rank_vetorial": idx,
            "source": doc.metadata["source"],
            "chunk_id": doc.metadata["chunk_id"],
            "trecho": doc.page_content.replace("\n", " ")[:220],
        }
        for idx, doc in enumerate(candidate_docs, start=1)
    ]
)

candidate_df


,rank_vetorial,source,chunk_id,trecho
0,1,04_manual_compras,16,"# Manual de Compras e Contratacao ## Objetivo Este manual define como comprar software, hardware e servicos com controle financeiro e compliance."
1,2,04_manual_compras,19,"## Fornecedores Novos fornecedores precisam enviar dados bancarios, contrato social e aceite do codigo de conduta. Compras sem cadastro previo nao devem se..."
2,3,04_manual_compras,18,"## Software Licencas de software SaaS devem ter owner interno, centro de custo e prazo contratual definidos. Renovacoes automaticas precisam de revalidacao..."
3,4,04_manual_compras,17,## Faixas de aprovacao - Ate R$ 1.000: lider de time. - De R$ 1.001 a R$ 5.000: gerente da area. - Acima de R$ 5.000 e ate R$ 20.000: diretor da area + com...
4,5,01_politica_reembolso,0,"# Politica de Reembolso e Despesas ## Objetivo Esta politica define quais gastos corporativos podem ser reembolsados, quais comprovantes sao obrigatorios ..."
5,6,02_politica_viagens,8,"## Reserva Passagens e hotel devem ser emitidos pela plataforma corporativa. Quando nao houver tarifa disponivel na plataforma, o time administrativo pode ..."


In [25]:
reranker_model_name = os.getenv(
    "RERANKER_MODEL",
    "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1",
)
reranker = CrossEncoder(reranker_model_name)

rerank_scores = reranker.predict([[query, doc.page_content] for doc in candidate_docs])
reranked_rows = []
for doc, score in sorted(zip(candidate_docs, rerank_scores), key=lambda pair: pair[1], reverse=True):
    reranked_rows.append(
        {
            "score_reranker_higher_is_better": round(float(score), 4),
            "source": doc.metadata["source"],
            "chunk_id": doc.metadata["chunk_id"],
            "trecho": doc.page_content.replace("\n", " ")[:220],
        }
    )

reranked_df = pd.DataFrame(reranked_rows)
reranked_df


,score_reranker_higher_is_better,source,chunk_id,trecho
0,2.3136,04_manual_compras,17,## Faixas de aprovacao - Ate R$ 1.000: lider de time. - De R$ 1.001 a R$ 5.000: gerente da area. - Acima de R$ 5.000 e ate R$ 20.000: diretor da area + com...
1,-6.7637,02_politica_viagens,8,"## Reserva Passagens e hotel devem ser emitidos pela plataforma corporativa. Quando nao houver tarifa disponivel na plataforma, o time administrativo pode ..."
2,-7.1413,04_manual_compras,19,"## Fornecedores Novos fornecedores precisam enviar dados bancarios, contrato social e aceite do codigo de conduta. Compras sem cadastro previo nao devem se..."
3,-7.3102,04_manual_compras,16,"# Manual de Compras e Contratacao ## Objetivo Este manual define como comprar software, hardware e servicos com controle financeiro e compliance."
4,-7.8811,04_manual_compras,18,"## Software Licencas de software SaaS devem ter owner interno, centro de custo e prazo contratual definidos. Renovacoes automaticas precisam de revalidacao..."
5,-9.1513,01_politica_reembolso,0,"# Politica de Reembolso e Despesas ## Objetivo Esta politica define quais gastos corporativos podem ser reembolsados, quais comprovantes sao obrigatorios ..."


In [26]:
top_contexts = reranked_rows[:3]
context = "\n\n".join(
    [
        f"[Fonte: {item['source']} | chunk {item['chunk_id']}]\n{item['trecho']}"
        for item in top_contexts
    ]
)

print(context)


[Fonte: 04_manual_compras | chunk 17]
## Faixas de aprovacao  - Ate R$ 1.000: lider de time. - De R$ 1.001 a R$ 5.000: gerente da area. - Acima de R$ 5.000 e ate R$ 20.000: diretor da area + compras. - Acima de R$ 20.000: comite de investimentos.

[Fonte: 02_politica_viagens | chunk 8]
## Reserva  Passagens e hotel devem ser emitidos pela plataforma corporativa. Quando nao houver tarifa disponivel na plataforma, o time administrativo pode autorizar compra manual.

[Fonte: 04_manual_compras | chunk 19]
## Fornecedores  Novos fornecedores precisam enviar dados bancarios, contrato social e aceite do codigo de conduta. Compras sem cadastro previo nao devem ser executadas.  ## Excecoes  Compras emergenciais de seguranca po


In [27]:
if os.getenv("OPENAI_API_KEY"):
    from openai import OpenAI

    client = OpenAI()
    prompt = f"""Você é um assistente de RAG interno.

Responda à pergunta usando apenas o contexto abaixo.
Se faltar evidência, diga explicitamente.
Cite a fonte entre colchetes.

Pergunta:
{query}

Contexto:
{context}
"""

    response = client.chat.completions.create(
        model=os.getenv("OPENAI_CHAT_MODEL", "gpt-4o-mini"),
        temperature=0,
        messages=[
            {"role": "system", "content": "Responda de forma objetiva e cite as fontes usadas."},
            {"role": "user", "content": prompt},
        ],
    )
    print(response.choices[0].message.content)
else:
    print("Sem OPENAI_API_KEY: pare no retrieval + reranking ou adicione a chave para a etapa de geração.")


As compras de software acima de R$ 5.000 e até R$ 20.000 são aprovadas pelo diretor da área e pelo setor de compras. Para compras acima de R$ 20.000, a aprovação é feita pelo comitê de investimentos [Fonte: 04_manual_compras | chunk 17].


## Como trocar o vector DB

A interface mental continua a mesma: **documento -> chunk -> embedding -> upsert -> search**.

### Chroma

```python
from langchain_chroma import Chroma

store = Chroma(
    collection_name="rag_docs",
    embedding_function=embeddings,
    persist_directory="./.chroma_lab/demo",
    collection_metadata={"hnsw:space": "cosine"},
)
store.add_documents(chunks)
store.similarity_search("minha query", k=5)
```

### pgvector

```sql
CREATE EXTENSION IF NOT EXISTS vector;

CREATE TABLE rag_chunks (
  id BIGSERIAL PRIMARY KEY,
  source TEXT,
  content TEXT,
  embedding VECTOR(384)
);

SELECT source, content
FROM rag_chunks
ORDER BY embedding <=> :query_embedding
LIMIT 5;
```

### Qdrant

```python
from qdrant_client import QdrantClient, models

client = QdrantClient(url="http://localhost:6333")
client.create_collection(
    collection_name="rag_docs",
    vectors_config=models.VectorParams(size=384, distance=models.Distance.COSINE),
)
client.upsert(
    collection_name="rag_docs",
    points=[
        models.PointStruct(id=idx, vector=vector, payload={"source": source, "text": text})
        for idx, (vector, source, text) in enumerate(records)
    ],
)
client.search(collection_name="rag_docs", query_vector=query_vector, limit=5)
```

### Pinecone

```python
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
pc.create_index(
    name="rag-docs",
    dimension=384,
    metric="cosine",
    spec=ServerlessSpec(cloud="aws", region="us-east-1"),
)
index = pc.Index("rag-docs")
index.upsert(
    vectors=[{"id": "1", "values": vector, "metadata": {"source": source, "text": text}}]
)
index.query(vector=query_vector, top_k=5, include_metadata=True)
```

### Weaviate

```python
import weaviate

client = weaviate.connect_to_local()
collection = client.collections.create(name="RagDocs", vectorizer_config=None)
collection.data.insert(properties={"source": source, "text": text}, vector=vector)
collection.query.near_vector(near_vector=query_vector, limit=5)
```

### Milvus

```python
from pymilvus import DataType, MilvusClient

client = MilvusClient(uri="http://localhost:19530")
schema = client.create_schema(auto_id=False, enable_dynamic_field=True)
schema.add_field(field_name="id", datatype=DataType.INT64, is_primary=True)
schema.add_field(field_name="source", datatype=DataType.VARCHAR, max_length=256)
schema.add_field(field_name="text", datatype=DataType.VARCHAR, max_length=4096)
schema.add_field(field_name="embedding", datatype=DataType.FLOAT_VECTOR, dim=384)
client.create_collection(collection_name="rag_docs", schema=schema)
client.insert(collection_name="rag_docs", data=[{"id": 1, "source": source, "text": text, "embedding": vector}])
client.search(collection_name="rag_docs", data=[query_vector], limit=5, output_fields=["source", "text"])
```


## Regra prática

- `Chroma`: demo local e protótipo
- `pgvector`: se já existe Postgres e o volume cabe
- `Qdrant`: open-source rápido, bom com filtros
- `Pinecone`: managed, zero-ops, paga premium
- `Weaviate`: forte em hybrid e ecossistema próprio
- `Milvus`: escala maior e operação mais pesada
